# DEG analysis - SlideTags dataset (Striatum)Mirrors `NucSeq_XDP/DEG/NucSeq_DEG_factors.ipynb`, adapted for`all_lib_adata_zoned_ct-cured_aligned.h5ad`. Differences from that notebook:- **Single tissue** (Striatum only) - no `TISSUE` branching needed.- **Condition levels** are `"healthy"`/`"diseased"` (not `Control`/`XDP`), and only **9 donors**  total: **2 healthy** (SCF-22-057, SCF-23-068) vs **7 diseased**. Healthy vs diseased  contrasts therefore have only 2 donors on the healthy side - see the warning printed before  that section.- **No `cohort` column**, and **`sex` is 100% male** across all 9 donors (zero variance) - so  neither is usable as a covariate here.- A **fourth `TYPE_DEG` option**, `"r_nebula"`, is added (via a one-line change to  `utils/nebula_with_factors.py`): it uses the *real*, spatially-measured dorsal-ventral  coordinate `r` (fit from Slide-tags x/y positions via splines), instead of the  gene-expression-derived `gradient_score` proxy used in the NucSeq notebook. `r` is only  available for donors that have a fitted spline file and for cells with spatial data  (`has_spatial=True`) - see the "Spatial position r" section, where you choose which donors  enter that specific analysis.- Two **independent, self-contained "Run" sections** at the bottom, matching how the NucSeq  notebook lays out "Run for all ct" vs "Run for all Zones" as separate blocks you run  selectively:  1. **Healthy vs diseased, per cell type** (loops every `ct_for_deg` category)  2. **Cell type vs cell type** (any two `ct_for_deg` categories, optionally restricted to one     condition, or pooled with `condition` added as a covariate)Both Run sections share one `TYPE_DEG` (set once below) and both automatically pick upwhichever extra covariate that implies (`gradient_score`, `r`, PCs, or none) - `TYPE_DEG`drives that internally in `nebula_with_factors.run_nebula_with_factors`, so it should **not**be added by hand to `COVARIATES_FOR_DEG` (adding it manually creates a duplicate term in theauxiliary factor-significance test).

In [ ]:
%load_ext autoreload%autoreload 2import pandas as pdimport numpy as npfrom scipy import sparseimport scanpy as scimport matplotlib.pyplot as pltimport osimport globimport rpy2.robjects as roimport shutilimport seaborn as snsfrom dotenv import load_dotenv; load_dotenv()from utils import preprocessingfrom utils import nebula_with_factors as nebula_utils# ATTENTION: need for proper running in jupyter notebook%matplotlib inlinePARALLEL_NEBULA_SCRIPT_PATH = os.getenv("PARALLEL_NEBULA_SCRIPT")ADATA_PATH = "/home/gdallagl/myworkdir/XDP/data/XDP/SlideTags_dataset/all_lib_adata_zoned_ct-cured_aligned.h5ad"# ATTENTION: single-tissue (Striatum) dataset -> no TISSUE if/elif needed here (unlike NucSeq_DEG_factors.ipynb)# TYPE_DEG controls which extra covariate/factor is modeled alongside the main contrast:#   "baseline_nebula"       -> no extra covariate#   "PCA_nebula"             -> healthy(baseline)-derived PCs, projected onto the rest, as covariates#   "gradient_score_nebula"  -> transcriptomic dorsal-ventral proxy score (works for ALL cells, incl. non-spatial ones)#   "r_nebula"                -> REAL spatial dorsal-ventral position `r` from fitted splines#                                (only cells/donors with spatial data + a fitted spline - see "Spatial position r" section)TYPE_DEG = "gradient_score_nebula"# name of cell type variable annotation to use in this analysisCT_FOR_DEG_VARIABLE = "ct_for_deg"# name of sample variableSAMPLE_VARIABLE = "donor_id"# variable to test if differentially expressed (healthy vs diseased contrast)CONTRAST_VARIABLE = "condition"CONTRAST_BASELINE = "healthy"CONTRAST_STIM = "diseased"# ATTENTION: unlike NucSeq_DEG_factors.ipynb, this dataset has no "cohort" column, and "sex" is# 100% male across all 9 donors (zero variance) -> neither can be used as a covariate hereCOVARIATES_FOR_DEG = ["age_at_death", "pct_counts_mt", "pct_intronic"]# library size col for nebulaLIBRARY_SIZE_COL = "total_counts"# Save folderDEG_FOLDER = f"/home/gdallagl/myworkdir/XDP/data/XDP/SlideTags_dataset/DEG/{TYPE_DEG}"os.makedirs(DEG_FOLDER, exist_ok=True)DEG_FOLDER

# Read adata

In [ ]:
print("Loading adata...")adata = sc.read_h5ad(ADATA_PATH)print(adata.obs.columns)############################ATTENTION: use gene names for DEGadata.var.index = adata.var["gene_symbol"]#######################################################ATTENTION: `condition` has a dirty "diseased " (trailing-space) level that exactly overlaps# with the cells that have no donor_id (114,692 unassigned demux-leftover cells) -> strip itadata.obs["condition"] = adata.obs["condition"].astype(str).str.strip()#######################################################ATTENTION: drop cells with no donor_id - can't fit the (1|donor) random effect for them, and# they're exactly the dirty-condition leftover cells above (a whole extra "sample_09" pool# that failed genotype demultiplexing)n_before = adata.n_obsadata = adata[adata.obs[SAMPLE_VARIABLE].notna()].copy()print(f"Dropped {n_before - adata.n_obs} cells with no {SAMPLE_VARIABLE} (unassigned demux leftovers)")#######################################################ATTENTION: remove "bad cell" types (same convention as NucSeq_DEG_factors.ipynb), including# the literal "nan" string category (a real categorical level here, not a missing value)mask_good_ct = (    (~adata.obs[CT_FOR_DEG_VARIABLE].astype(str).str.startswith("-")) &    (adata.obs[CT_FOR_DEG_VARIABLE].astype(str) != "nan"))adata = adata[mask_good_ct].copy()#######################################################ATTENTION: be sure cell types have no spaceadata.obs[CT_FOR_DEG_VARIABLE] = adata.obs[CT_FOR_DEG_VARIABLE].astype(str).str.replace(" ", "_")###########################print("\nDonors:")print(adata.obs.drop_duplicates(SAMPLE_VARIABLE)[[SAMPLE_VARIABLE, "condition", "age_at_death", "sex"]].sort_values(SAMPLE_VARIABLE))print("\nCell types:")print(adata.obs[CT_FOR_DEG_VARIABLE].value_counts())

# DV/gradient score (transcriptomic proxy)Only needed when `TYPE_DEG == "gradient_score_nebula"` - reuses the same BICAN spatialgene-correlation files as `NucSeq_DEG_factors.ipynb`. This is a proxy score derived from geneexpression, so it works for every cell (spatial or not) - contrast with the real spatial `r`computed in the next section, which only covers cells with actual spatial coordinates.

In [ ]:
# Genes whose Matrix/Patch/eMSN spatial gradient (health vs XDP, from the BICAN spatial-spline# analysis) is strong AND agrees in sign in BOTH conditions - i.e. disease-invariant spatial# markers. Deliberately NOT genes whose gradient differs between health/XDP: those are# candidate DEG hits themselves, and scoring cells with the genes being tested would regress# a gene's DE signal out against itself.CORR_DIR = "/home/gdallagl/myworkdir/XDP/data/BICAN/bican_spatial_zonated/genes_correlation"STABLE_GRAD_THR           = 0.2   # minimum |rho| in EACH condition to count as "has gradient"STABLE_GRAD_PADJ_THR      = 0.05STABLE_GRAD_MIN_PCT_EXPR  = 0.20STABLE_GRAD_MIN_MEAN_EXPR = 0.10df_corr_health = pd.read_csv(f"{CORR_DIR}/df_corr.csv", index_col=0)df_corr_xdp    = pd.read_csv(f"{CORR_DIR}/df_corr_xdp.csv", index_col=0)common_genes   = df_corr_health.index.intersection(df_corr_xdp.index)stable_grad_weight = {}   # ct -> Series(gene -> weight), weight = mean(rho_health, rho_XDP)for ct in ["Matrix", "Patch", "eMSN"]:    dfh, dfx = df_corr_health.loc[common_genes], df_corr_xdp.loc[common_genes]    rho_h, rho_x = dfh[f"rho_{ct}"], dfx[f"rho_{ct}"]    reliable_h = (dfh[f"padj_{ct}"] < STABLE_GRAD_PADJ_THR) & (dfh[f"pct_expr_{ct}"] >= STABLE_GRAD_MIN_PCT_EXPR) & (dfh[f"mean_expr_{ct}"] >= STABLE_GRAD_MIN_MEAN_EXPR)    reliable_x = (dfx[f"padj_{ct}"] < STABLE_GRAD_PADJ_THR) & (dfx[f"pct_expr_{ct}"] >= STABLE_GRAD_MIN_PCT_EXPR) & (dfx[f"mean_expr_{ct}"] >= STABLE_GRAD_MIN_MEAN_EXPR)    stable = (        (rho_h.abs() >= STABLE_GRAD_THR) & (rho_x.abs() >= STABLE_GRAD_THR) &        (np.sign(rho_h) == np.sign(rho_x)) &        reliable_h & reliable_x    ).fillna(False)    genes_ct = common_genes[stable]    stable_grad_weight[ct] = pd.concat([rho_h[genes_ct], rho_x[genes_ct]], axis=1).mean(axis=1)    print(f"{ct}: {len(genes_ct)} stable gradient genes (agree in sign, health & XDP)")

In [ ]:
def calc_gradient_score(adata, gene_weights, ct_col="ct_for_deg", score_col="gradient_score"):    """Score every cell by its predicted position along the Matrix/Patch/eMSN dorsal-ventral    spatial gradient, using per-cell-type gene weights (dict: ct -> Series[gene -> weight],    weight = mean spatial rho in health & XDP - see cell above).    ATTENTION: adata.X must be raw counts.    ATTENTION: cells whose ct_col value isn't a key of `gene_weights` get NaN (e.g. glia).    Each gene is z-scored across cells of its own cell type before being weighted by rho, so    a highly-expressed-but-weakly-correlated gene can't dominate a lowly-expressed-but-    strongly-correlated one purely by scale - only how informative the gene actually is about    position (its rho) determines its contribution to the score.    """    sc.pp.normalize_total(adata, target_sum=1e4)    sc.pp.log1p(adata)    adata.obs[score_col] = np.nan    for ct, weight in gene_weights.items():        cell_mask = (adata.obs[ct_col] == ct).values        if cell_mask.sum() == 0:            continue        genes_here = [g for g in weight.index if g in adata.var_names]        print(f"{ct}: scoring {cell_mask.sum()} cells with {len(genes_here)}/{len(weight)} gradient genes found in adata")        X = adata[cell_mask, genes_here].X        X = X.toarray() if sparse.issparse(X) else np.asarray(X)        X = (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-9)   # z-score per gene, within this cell type        adata.obs.loc[cell_mask, score_col] = X @ weight.loc[genes_here].values    return adata

In [ ]:
# Gene weights above are keyed by macro-compartment (Matrix/Patch/eMSN), not by D1/D2 subtype -# map each ct_for_deg category onto the right weight vector. Categories with no entry here# (eMSN_NUDAP, eMSN_StrioMat, OT_D1_ICj, non-MSN, glia, ...) get NaN gradient_score - expected.CT_TO_GRADIENT_GROUP = {    "Matrix_D1": "Matrix", "Matrix_D2": "Matrix",    "Patch_D1": "Patch", "Patch_D2": "Patch",    "eMSN_D1D2": "eMSN",}adata.obs["ct_for_gradient"] = adata.obs[CT_FOR_DEG_VARIABLE].map(CT_TO_GRADIENT_GROUP)if TYPE_DEG == "gradient_score_nebula":    save_csv_path = f"{os.path.dirname(ADATA_PATH)}/gradient_scores.csv"    if not os.path.exists(save_csv_path):        adata.X = adata.layers["counts"].copy()  # use raw counts for score calculation        adata = calc_gradient_score(adata, stable_grad_weight, ct_col="ct_for_gradient", score_col="gradient_score")        adata.obs[["gradient_score"]].to_csv(save_csv_path)    else:        df_scores = pd.read_csv(save_csv_path, index_col=0)        adata.obs = adata.obs.merge(df_scores, how="left", left_index=True, right_index=True, validate="one_to_one")    # only cells mapped to a gradient group are expected to have a score - everything else    # (glia, eMSN_NUDAP/StrioMat, OT_D1_ICj, non-MSN, ...) stays NaN, by design    assert not adata.obs.loc[adata.obs["ct_for_gradient"].notna(), "gradient_score"].isna().any(), \        "Some Matrix/Patch/eMSN cells have no gradient score!"    print(adata.obs.groupby("ct_for_gradient")["gradient_score"].describe())else:    print(f"TYPE_DEG={TYPE_DEG!r}: skipping gradient_score computation (not needed for this mode)")

# Spatial position r (real dorsal-ventral coordinate)Only needed when `TYPE_DEG == "r_nebula"`. Reads the per-donor spline fits already computed in`bican_spatial_zonated/for_splines/{donor_id}_x_y_with_splines.csv` (same pipeline/conventionas `gradients_in_STR/find_spatial_grad_gene_MSN_types_refactored.ipynb`): `r` = position alongthe fitted dorsal-ventral spline, normalized to each donor's own max; `d` = perpendiculardistance off the spline axis (not used here, kept for reference/QC).**Not every donor has a fitted spline yet** - `SAMPLES_FOR_R` below defaults to whicheverdonors already have a `*_x_y_with_splines.csv` file. Edit that list by hand to add/removedonors from the `r_nebula` analysis (e.g. once a new spline is fit, or to exclude a donorwhose fit looks unreliable).

In [ ]:
SPLINES_DIR = "/home/gdallagl/myworkdir/XDP/data/BICAN/bican_spatial_zonated/for_splines"available_spline_donors = sorted({    os.path.basename(p).replace("_x_y_with_splines.csv", "")    for p in glob.glob(f"{SPLINES_DIR}/*_x_y_with_splines.csv")    if os.path.basename(p).startswith("SCF-")})print("Donors with a fitted spline:", available_spline_donors)donors_without_spline = sorted(set(adata.obs[SAMPLE_VARIABLE].dropna().unique()) - set(available_spline_donors))if donors_without_spline:    print(f"ATTENTION: {donors_without_spline} have no fitted spline yet (no *_x_y_with_splines.csv) "          f"-> will get NaN r, excluded from any r_nebula run below")# EDIT this list to control which donors can enter an r_nebula analysisSAMPLES_FOR_R = available_spline_donors

In [ ]:
adata.obs["r"] = np.nanadata.obs["r_raw"] = np.nanadata.obs["d"] = np.nanif TYPE_DEG == "r_nebula":    for donor in SAMPLES_FOR_R:        csv_path = f"{SPLINES_DIR}/{donor}_x_y_with_splines.csv"        if not os.path.exists(csv_path):            print(f"{donor}: spline file not found, skipping")            continue        df_spline = pd.read_csv(csv_path, index_col=0)        df_spline.index = df_spline.index.astype(str)        df_spline["r_raw"] = df_spline["r"].copy()          # keep original for reference        df_spline["r"] = df_spline["r"] / df_spline["r"].max()  # normalize per donor        donor_barcodes = adata.obs_names[adata.obs[SAMPLE_VARIABLE] == donor]        common_idx = df_spline.index.intersection(donor_barcodes)        missing = df_spline.index.difference(donor_barcodes)        print(f"{donor}: {len(common_idx)}/{len(df_spline)} spline barcodes matched in adata ({len(missing)} not found, likely QC-filtered)")        adata.obs.loc[common_idx, "r"] = df_spline.loc[common_idx, "r"].values        adata.obs.loc[common_idx, "r_raw"] = df_spline.loc[common_idx, "r_raw"].values        adata.obs.loc[common_idx, "d"] = df_spline.loc[common_idx, "d"].values    print(f"\nTotal cells with a valid r: {adata.obs['r'].notna().sum()} / {adata.n_obs}")    print(adata.obs.groupby(SAMPLE_VARIABLE)["r"].apply(lambda s: s.notna().sum()))else:    print(f"TYPE_DEG={TYPE_DEG!r}: skipping r merge (not needed for this mode)")

# Define run helper

In [ ]:
def run_nebula_for_group(adata_group, subfolder, contrast_variable, contrast_baseline, contrast_stim, covariates):    """Thin wrapper around nebula_utils.run_nebula_with_factors: handles folder naming,    stale-tmp cleanup, and setting .X to raw counts. TYPE_DEG (global) decides which extra    covariate (gradient_score / r / PCs / none) gets added internally - do not add it here.    """    adata_group = adata_group.copy()    adata_group.X = adata_group.layers["counts"].copy()    ADATA_PATH_QS = f"{DEG_FOLDER}/{subfolder}/tmp_{os.path.splitext(os.path.basename(ADATA_PATH))[0]}_{subfolder}_for_nebula.qs"    SAVE_RESULT = os.path.dirname(ADATA_PATH_QS)    os.makedirs(SAVE_RESULT, exist_ok=True)    print(ADATA_PATH_QS)    # Delete stale nebula chunks so chunk-sobj-by-genes.R doesn't reuse files from a previous    # failed run (which may have had different covariate columns)    _nebula_dir = f"{os.path.splitext(ADATA_PATH_QS)[0]}__nebula_ln"    if os.path.exists(_nebula_dir):        shutil.rmtree(_nebula_dir)    nebula_utils.run_nebula_with_factors(        adata_group,        ADATA_PATH_QS, SAVE_RESULT, PARALLEL_NEBULA_SCRIPT_PATH,        contrast_variable, contrast_baseline, contrast_stim,        SAMPLE_VARIABLE, LIBRARY_SIZE_COL, covariates,        N_PCS_TO_USE=None,  # default knee method        variance_threshold=None, delete_tmp_files=True, TYPE_DEG=TYPE_DEG,        MIN_SAMPLES_PER_CONDITION=4, MIN_NUMBER_GENES_PER_ANALYSIS=3000,        CNMF_SPECTRA_PATH=None,    )

# Run: healthy vs diseased, per cell typeLoops every `ct_for_deg` category and runs the healthy-vs-diseased contrast within it (mirrors`NucSeq_DEG_factors.ipynb`'s "Run for all ct" section).

In [ ]:
print("ATTENTION: only 2 donors are 'healthy' in this dataset (SCF-22-057, SCF-23-068) vs "      "7 'diseased' -> MIN_SAMPLES_PER_CONDITION=4 below means EVERY cell type here will be "      "skipped for this contrast (healthy n_donors=2 < 4). Lower MIN_SAMPLES_PER_CONDITION in "      "run_nebula_for_group if you want to force it through anyway - results would be very "      "fragile with only 2 donors on one side (no real donor-level variance estimate).")for ct in sorted(adata.obs[CT_FOR_DEG_VARIABLE].unique().tolist()):    print(f"\n\n######################\n{ct}\n######################\n")    adata_ct = adata[adata.obs[CT_FOR_DEG_VARIABLE] == ct].copy()    if TYPE_DEG == "r_nebula":        adata_ct = adata_ct[adata_ct.obs["r"].notna()].copy()        print(f"r_nebula mode: {adata_ct.n_obs} cells with valid r remain")    print(adata_ct.obs.groupby(CONTRAST_VARIABLE)[SAMPLE_VARIABLE].nunique())    run_nebula_for_group(adata_ct, ct, CONTRAST_VARIABLE, CONTRAST_BASELINE, CONTRAST_STIM, list(COVARIATES_FOR_DEG))

# Run: cell type vs cell typeCompares any two `ct_for_deg` categories against each other. Optionally restrict to onecondition (`CONDITION_FILTER = "healthy"` or `"diseased"`), or pool both and let `condition`enter as a covariate (`CONDITION_FILTER = None`).

In [ ]:
CT_BASELINE = "Matrix_D1"CT_STIM = "Matrix_D2"# Restrict to one condition before comparing cell types, or None to pool both and add# `condition` itself as a covariate instead.#   "healthy"  -> only healthy donors#   "diseased" -> only diseased donors#   None       -> keep both, condition becomes a covariateCONDITION_FILTER = Noneadata_ctc = adata[adata.obs[CT_FOR_DEG_VARIABLE].isin([CT_BASELINE, CT_STIM])].copy()if CONDITION_FILTER is not None:    adata_ctc = adata_ctc[adata_ctc.obs[CONTRAST_VARIABLE] == CONDITION_FILTER].copy()if TYPE_DEG == "r_nebula":    adata_ctc = adata_ctc[adata_ctc.obs["r"].notna()].copy()    print(f"r_nebula mode: {adata_ctc.n_obs} cells with valid r remain")print(adata_ctc.obs.groupby([CT_FOR_DEG_VARIABLE, CONTRAST_VARIABLE]).size())covariates = list(COVARIATES_FOR_DEG)if CONDITION_FILTER is None:    covariates.append(CONTRAST_VARIABLE)  # "condition" as covariate since both are pooledsubfolder = f"{CT_BASELINE}_vs_{CT_STIM}" + (f"_{CONDITION_FILTER}" if CONDITION_FILTER else "_allConditions")run_nebula_for_group(adata_ctc, subfolder, CT_FOR_DEG_VARIABLE, CT_BASELINE, CT_STIM, covariates)